# Day 28 — Kaggle vLLM evidence (IP07)

This notebook runs **only** the GPU inference extension. Kafka, Airflow, Delta, Feast, Qdrant, MLflow, Docker Compose, and the end-to-end trace remain on the local platform.

Before running: choose a **T4** GPU in Kaggle Settings, enable Internet, and make sure the session has quota. Do not use P100 as the baseline for this lab. Kaggle sessions are temporary; save the generated JSON before the session ends.

The notebook deliberately does not create a public tunnel or contain tunnel credentials. A real deployment must use an approved tunnel/reverse proxy with authentication and access restriction.

## Installation on Kaggle

Kaggle's managed image does not include `ensurepip`, so Python virtual environments cannot be created reliably. Installing vLLM in the active notebook kernel can produce dependency-resolver warnings for unrelated base packages (for example Gradio or Google SDKs). Those warnings are recorded but do not by themselves mean vLLM failed; the next cell verifies the installed package and CUDA before the server is started.

In [ ]:
from pathlib import Path

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
VLLM_VERSION = "0.26.0"
PORT = 8000
MAX_MODEL_LEN = 4096
GPU_MEMORY_UTILIZATION = 0.85
WORKDIR = Path("/kaggle/working")
LOG_PATH = WORKDIR / "vllm.log"
EVIDENCE_PATH = WORKDIR / "ip07-kaggle-server-evidence.json"

print({"model_id": MODEL_ID, "vllm_version": VLLM_VERSION, "workdir": str(WORKDIR)})

In [ ]:
import subprocess
import sys

subprocess.run(["nvidia-smi"], check=True)

probe = subprocess.run(
    [sys.executable, "-c", "import torch; print(torch.cuda.is_available()); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')"],
    check=True,
    capture_output=True,
    text=True,
).stdout.splitlines()

if not probe or probe[0].strip() != "True":
    raise RuntimeError("CUDA is unavailable. In Kaggle Settings choose Accelerator: GPU, then restart the session.")
gpu_name = probe[1].strip() if len(probe) > 1 else "unknown"
if "P100" in gpu_name.upper():
    raise RuntimeError(f"{gpu_name} is not the Day 28 baseline. Stop and select a T4 session instead.")
print(f"GPU accepted for this notebook: {gpu_name}")

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-cache-dir", f"vllm=={VLLM_VERSION}"],
    check=True,
)

subprocess.run(
    [sys.executable, "-c", "import torch, vllm; print('vllm=', vllm.__version__); print('torch=', torch.__version__); print('cuda=', torch.version.cuda); print('gpu=', torch.cuda.get_device_name(0))"],
    check=True,
)

## Start vLLM

Kaggle rejects shell background commands such as `!command &`. `subprocess.Popen` starts the server without that unsupported shell path. Run this cell once per Kaggle session.

In [ ]:
import os
import shutil
import site
import subprocess

if "VLLM_PROCESS" in globals() and VLLM_PROCESS.poll() is None:
    raise RuntimeError(f"vLLM is already running with PID {VLLM_PROCESS.pid}")

vllm_executable = shutil.which("vllm")
if not vllm_executable:
    raise RuntimeError("vllm CLI was not installed on PATH; inspect the preceding pip output.")

environment = os.environ.copy()
cuda_library_dirs = [
    str(path)
    for package_root in site.getsitepackages()
    for path in Path(package_root).glob("nvidia/*/lib")
    if path.is_dir()
]
if not cuda_library_dirs:
    raise RuntimeError("vLLM CUDA runtime libraries were not installed.")
environment["LD_LIBRARY_PATH"] = ":".join(cuda_library_dirs + [environment.get("LD_LIBRARY_PATH", "")])
print(f"Configured {len(cuda_library_dirs)} CUDA library directories for vLLM")
environment["HF_HOME"] = str(WORKDIR / "hf-cache")
command = [
    vllm_executable, "serve", MODEL_ID,
    "--host", "0.0.0.0",
    "--port", str(PORT),
    "--dtype", "half",
    "--max-model-len", str(MAX_MODEL_LEN),
    "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),
]

log_file = LOG_PATH.open("w", buffering=1)
VLLM_PROCESS = subprocess.Popen(
    command,
    stdout=log_file,
    stderr=subprocess.STDOUT,
    env=environment,
    start_new_session=True,
)
print(f"vLLM PID: {VLLM_PROCESS.pid}")
print(f"Log file: {LOG_PATH}")

In [ ]:
import time
import requests

base_url = f"http://127.0.0.1:{PORT}"
deadline = time.monotonic() + 600
last_error = "server has not answered yet"
while time.monotonic() < deadline:
    if VLLM_PROCESS.poll() is not None:
        raise RuntimeError(f"vLLM exited with {VLLM_PROCESS.returncode}\n{LOG_PATH.read_text(errors='replace')[-6000:]}")
    try:
        health = requests.get(f"{base_url}/health", timeout=5)
        if health.status_code == 200:
            break
        last_error = f"/health returned {health.status_code}"
    except requests.RequestException as error:
        last_error = f"{type(error).__name__}: {error}"
    time.sleep(5)
else:
    raise TimeoutError(f"vLLM did not become ready: {last_error}\n{LOG_PATH.read_text(errors='replace')[-6000:]}")

print("vLLM health check passed")

In [ ]:
import json
import re
import requests

version_response = requests.get(f"{base_url}/version", timeout=20)
version_response.raise_for_status()
try:
    version = version_response.json()
except ValueError:
    version = version_response.text.strip()

models_response = requests.get(f"{base_url}/v1/models", timeout=20)
models_response.raise_for_status()
models = models_response.json()
served_models = [item["id"] for item in models.get("data", [])]
if MODEL_ID not in served_models:
    raise AssertionError(f"configured model {MODEL_ID!r} is absent from {served_models!r}")

metrics_response = requests.get(f"{base_url}/metrics", timeout=20)
metrics_response.raise_for_status()
metric_names = sorted(set(re.findall(r"(?:^|\n)(?:# (?:HELP|TYPE) )?(vllm:[A-Za-z0-9_:]+)", metrics_response.text)))
if not metric_names:
    raise AssertionError("/metrics has no vllm: metric names; this is not acceptable IP07 evidence")

server_identity = {
    "version": version,
    "served_models": served_models,
    "vllm_metric_names": metric_names[:20],
}
print(json.dumps(server_identity, indent=2))

In [ ]:
import json
import uuid
import requests

client_trace_id = uuid.uuid4().hex
completion = requests.post(
    f"{base_url}/v1/chat/completions",
    headers={"traceparent": f"00-{client_trace_id}-{uuid.uuid4().hex[:16]}-01"},
    json={
        "model": MODEL_ID,
        "messages": [{"role": "user", "content": "Reply with exactly: vLLM IP07 ready"}],
        "temperature": 0,
        "max_tokens": 16,
    },
    timeout=120,
)
completion.raise_for_status()
body = completion.json()
answer = body["choices"][0]["message"]["content"]
if not answer.strip():
    raise AssertionError("vLLM returned an empty completion")

smoke = {
    "client_trace_id": client_trace_id,
    "response_model": body.get("model"),
    "answer": answer,
}
print(json.dumps(smoke, indent=2))
print("This proves the Kaggle server. The local Day 28 API must be tested separately for end-to-end trace propagation.")

In [ ]:
import json
from datetime import datetime, timezone

evidence = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "is_real_vllm": True,
    "configured_model_id": MODEL_ID,
    **server_identity,
    "smoke": smoke,
    "notes": [
        "Kaggle URL and tunnel credentials are intentionally absent.",
        "End-to-end Day 28 trace/model-release evidence must be collected from the local platform after a protected tunnel is configured.",
    ],
}
EVIDENCE_PATH.write_text(json.dumps(evidence, indent=2), encoding="utf-8")
print(f"Wrote {EVIDENCE_PATH}")

## Connect to the local Docker platform — manual security gate

Kaggle does not make this notebook's `127.0.0.1:8000` available to your laptop by default. Configure an approved outbound tunnel or reverse proxy separately, protecting it with provider-level authentication and access restriction. Do **not** add a temporary URL or token to this notebook, Git, or the evidence JSON.

Once a protected base URL is available, set these only in the **local terminal** (not Git):

```bash
export LAB28_VLLM_BASE_URL='https://YOUR-PROTECTED-HOST/v1'
export LAB28_VLLM_MODEL_ID='Qwen/Qwen3-4B-Instruct-2507'
export LAB28_VLLM_REQUIRE_REAL=true
docker compose --env-file ports.template --profile full up -d --force-recreate api
uv run lab28 evidence
uv run pytest integration-tests/test_j1_golden_path.py -q
uv run pytest integration-tests/test_j3_promotion_rollback.py -q
uv run pytest integration-tests/test_j4_degraded_recovery.py -q
```

Only the local integration tests prove the final required response fields: trace ID, model ID, and MLflow release version.